# Transient transport: spatial refinement and time refinement

This lesson isolates the spatial approximation of conservative transient MHM. The existing [exponential-decay lesson](transient_variational.ipynb) separately verifies first-order backward Euler. Here the exact concentration is linear in time, so the continuum backward difference is exact; refine the macro mesh without hiding an accumulated temporal truncation error.

## 1. Choose the physical solution and derive the source

On the unit square, use capacity and diffusion one, $\beta=(1/4,-1/2)$, reaction $1/2$, zero exterior concentration, and zero initial concentration.

$$
\begin{aligned}
\partial_t c-\Delta c+\beta\cdot\nabla c+\tfrac12c&=f,\\
 c(t,x,y)&=t\phi(x,y),\qquad\phi=\sin(\pi x)\sin(\pi y),\\
 f(t,x,y)&=\phi+t[(2\pi^2+\tfrac12)\phi+\beta\cdot\nabla\phi].
\end{aligned}
$$

The divergence of the constant convection is zero. The source follows by analytical differentiation of the stated solution, before assembling any matrix. The transport convention uses half-advection pseudoflux; the physical transported flux is $\beta c-\nabla c$.

## 2. Write the local and global variational equations

The cell below defines every equation used in this study. `volume_forms` follows the displayed strong problem. Its skew convection form supplies the half-advection interface convention. The global equation sets interior concentration-jump and zero exterior moments to zero. One constant is a retained coarse mode; positive mass and reaction mean it is not a local nullspace.

Use local P2 on four fine triangles per macrotriangle, with one P1 trace per macroface. The binding owns geometric orientations and native coefficient ordering. Refining $H$ preserves all these degrees and partitions.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download checksum-verified support files; extraction does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/dc9d2a87ad868719822e1be1e3935a163fac0dac4620be5dfc00c23bfe9e7814/spatial_refinement-companion.zip"
COMPANION_SHA256 = "dc9d2a87ad868719822e1be1e3935a163fac0dac4620be5dfc00c23bfe9e7814"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prepare only this notebook's declared inputs, without executing its equations.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("transport/spatial_refinement.ipynb", directory=ROOT)


In [ ]:
import numpy as np
import ufl
import basix
import basix.ufl
from functools import partial
from time import perf_counter
from pymhm import (
    TriangleMesh, FaceSpace, SkeletonSpace, MeshHierarchy, LocalContext,
    Equation, columns, compile_form, bind_interface, bind_problem, assemble,
    MultiscaleProblem, ExecutionConfig, solve,
)
from pymhm.core.online import OfflineMultiscaleSystem
from pymhm.fem.scalar.operators import triangle_quadrature
from pymhm.linalg.linear import factorize
from pymhm.backends.spaces import bind_space, coefficient_map
from pymhm.fem.scalar.triangle import nodal_space


def exact_concentration(points, time):
    """Evaluate the independently manufactured concentration c=t*sin(pi*x)*sin(pi*y)."""
    return time*np.sin(np.pi*points[:,0])*np.sin(np.pi*points[:,1])


def exact_gradient(points, time):
    """Differentiate the concentration independently of finite-element tabulation."""
    return time*np.pi*np.column_stack((
        np.cos(np.pi*points[:,0])*np.sin(np.pi*points[:,1]),
        np.sin(np.pi*points[:,0])*np.cos(np.pi*points[:,1]),
    ))


def volume_forms(V, domain, dt):
    """State mass, conservative transport and the two independently derived source profiles."""
    c,v=ufl.TrialFunction(V),ufl.TestFunction(V)
    x=ufl.SpatialCoordinate(domain)
    beta=ufl.as_vector((0.25,-0.5))
    phi=ufl.sin(np.pi*x[0])*ufl.sin(np.pi*x[1])
    grad_phi=np.pi*ufl.as_vector((ufl.cos(np.pi*x[0])*ufl.sin(np.pi*x[1]),
                                ufl.sin(np.pi*x[0])*ufl.cos(np.pi*x[1])))
    spatial_force=(2*np.pi**2+0.5)*phi+ufl.dot(beta,grad_phi)
    dx=ufl.Measure('dx',domain=domain,metadata={'quadrature_degree':12})
    mass=c*v*dx
    a=((1/dt+0.5)*c*v+ufl.inner(ufl.grad(c),ufl.grad(v))
       +0.5*(ufl.dot(beta,ufl.grad(c))*v-c*ufl.dot(beta,ufl.grad(v))))*dx
    return a,mass,phi*v*dx,spatial_force*v*dx,v*dx


def local_time_equations(local: LocalContext, dt: float):
    """Declare local UFL backward Euler and negative concentration-jump moments."""
    native=local.native_space(degree=2)
    V=native.space
    c,v=ufl.TrialFunction(V),ufl.TestFunction(V)
    a,mass,constant_force,linear_force,mean=volume_forms(V,native.mesh,dt)
    M,F0,F1=compile_form(mass),compile_form(constant_force),compile_form(linear_force)
    b=local.trace_pairings(lambda phi,ds:phi*v*ds)
    balance=local.trace_pairings(lambda phi,ds:-phi*c*ds,axis='rows')
    local.field('concentration',native)
    return local.equations(
        a=a,L=F0+dt*F1,b=b,c=balance,
        coarse_basis=np.ones((native.size,1)),moments=columns(mean),
        metadata={'mass':M,'constant_force':F0,'linear_force':F1},
    )


def trajectory(n,dt,final_time,*,workers=1):
    """Construct the hierarchy, assemble once and advance the declared equations."""
    macro=TriangleMesh.unit_square(n)
    skeleton=SkeletonSpace(macro,tuple(FaceSpace.uniform(1) for _ in macro.faces))
    hierarchy=MeshHierarchy(macro,tuple(macro.submesh(i,2) for i in range(len(macro.cells))))
    interface=bind_interface(skeleton,convention='normal')
    problem=bind_problem(hierarchy,interface,partial(local_time_equations,dt=dt),
                         global_equation=Equation(0,0),retained=1)
    execution=ExecutionConfig('process' if workers>1 else 'serial',workers=workers,native_threads=1)
    system=assemble(problem,execution=execution)
    metadata=system.local_metadata
    previous=tuple(np.zeros(len(record['constant_force'])) for record in metadata)
    steps=round(final_time/dt)
    if not np.isclose(steps*dt,final_time):raise ValueError('dt must divide final time')
    residual=0.0
    with OfflineMultiscaleSystem(system) as offline,factorize(system.matrix) as global_factor:
        for step in range(1,steps+1):
            time=step*dt
            loads=tuple(record['constant_force']+time*record['linear_force']+record['mass']@old/dt
                        for record,old in zip(metadata,previous,strict=True))
            state=offline.with_loads(loads).solve(factorization=global_factor)
            residual=max(residual,float(state.raw_residual))
            previous=state.fields
    return macro,state,residual


def physical_errors(state,final_time,order):
    """Integrate concentration and broken-gradient errors with explicit fine-cell ownership."""
    bary,weights=triangle_quadrature(order)
    totals=np.zeros(2,dtype=np.longdouble)
    for field in state.field('concentration'):
        fine=field.mesh
        points=np.einsum('qi,tia->tqa',bary,fine.points[fine.cells])
        owners=np.repeat(np.arange(len(fine.cells)),len(bary))
        value,gradient=field.values_and_gradient(points.reshape(-1,2),cells=owners)
        value=value.reshape(points.shape[:2])-exact_concentration(points.reshape(-1,2),final_time).reshape(points.shape[:2])
        gradient=gradient.reshape(points.shape)-exact_gradient(points.reshape(-1,2),final_time).reshape(points.shape)
        for index,integrand in enumerate((value**2,np.sum(gradient**2,axis=-1))):
            totals[index]+=np.sum(fine.areas[:,None]*weights*integrand,dtype=np.longdouble)
    return dict(zip(('concentration_l2','broken_gradient_l2'),np.sqrt(totals).astype(float).tolist(),strict=True))


def spatial_row(n,dt,final_time,*,workers=1):
    """Acquire unchanged spaces and compare independent error quadrature orders."""
    started=perf_counter()
    macro,state,residual=trajectory(n,dt,final_time,workers=workers)
    norms={str(order):physical_errors(state,final_time,order) for order in (6,8)}
    low,high=norms.values()
    changes={name:abs(low[name]-high[name])/max(high[name],np.finfo(float).tiny) for name in high}
    return {'n':n,'H':np.sqrt(2)/n,'macro_cells':len(macro.cells),'dt':dt,'final_time':final_time,
            **high,'global_compatibility_relative_max':residual,
            'quadrature':{'assembly_degree':12,'error_orders':[6,8],'errors':norms,'relative_changes':changes},
            'wall_seconds':perf_counter()-started,
            'accepted':bool(residual<=1e-10 and max(changes.values())<=1e-6)}


def classical_row(n,dt,final_time,*,sample_points=None):
    """Independently assemble conforming global P2 UFL with strong zero concentration."""
    from dolfinx import fem
    started=perf_counter()
    mesh=TriangleMesh.unit_square(n)
    native=bind_space(mesh,basix.ufl.element('Lagrange','triangle',2,
                                           lagrange_variant=basix.LagrangeVariant.equispaced))
    try:
        a,mass,f0,f1,_=volume_forms(native.space,native.space.mesh,dt)
        A,M,F0,F1=compile_form(a),compile_form(mass),compile_form(f0),compile_form(f1)
        _,points=nodal_space(mesh,2)
        boundary=coefficient_map(native)[np.flatnonzero(np.any(np.isclose(points,0)|np.isclose(points,1),axis=1))]
        free=np.setdiff1d(np.arange(native.size),boundary)
        values=np.zeros(native.size)
        original=0.0
        with factorize(A[free][:,free]) as factor:
            for step in range(1,round(final_time/dt)+1):
                rhs=F0+step*dt*F1+M@values/dt
                values[free]=factor.solve(rhs[free])
                original=max(original,float(np.linalg.norm((A@values-rhs)[free])/np.linalg.norm(rhs[free])))
        numerical=fem.Function(native.space)
        numerical.x.array[:]=values
        x=ufl.SpatialCoordinate(native.space.mesh)
        truth=final_time*ufl.sin(np.pi*x[0])*ufl.sin(np.pi*x[1])
        dx=ufl.Measure('dx',domain=native.space.mesh,metadata={'quadrature_degree':16})
        delta=numerical-truth
        errors={'concentration_l2':float(np.sqrt(fem.assemble_scalar(fem.form(delta**2*dx)))),
                'broken_gradient_l2':float(np.sqrt(fem.assemble_scalar(fem.form(ufl.inner(ufl.grad(delta),ufl.grad(delta))*dx))))}
        record={'n':n,'dofs':native.size,'dt':dt,'final_time':final_time,**errors,
                'original_equations_relative_max':original,'wall_seconds':perf_counter()-started,
                'accepted':bool(original<=1e-10)}
        if sample_points is not None:
            record['sample_values']=native.evaluate(values,sample_points)
        return record
    finally:
        native.close()


## 3. Assemble, solve and inspect the physical fields

Run the locked `introduction` environment. These three statements expose the workflow: construct the hierarchy and equations, assemble the fixed time-step operator, solve each updated source, then evaluate named concentration fields. The scientific acquisition helper exports the exact preceding cell to spawn workers; it supplies no alternate PDE formula.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from examples.introduction.vector import plot_field_panels

macro,state,residual=trajectory(8,0.01,0.5)
fields=state.field('concentration')
points=[];triangles=[];values=[];offset=0
for field in fields:
    fine=field.mesh
    points.append(fine.points)
    triangles.append(fine.cells+offset)
    values.append(field.evaluate(fine.points))
    offset+=len(fine.points)
points=np.concatenate(points);triangles=np.concatenate(triangles);values=np.concatenate(values)
truth=exact_concentration(points,0.5)
classical=classical_row(64,0.01,0.5,sample_points=points)
reference_values=classical.pop('sample_values')
assert classical['accepted']
figure=plot_field_panels(macro,{
    'Analytical concentration':(points,triangles,truth),
    'MHM concentration':(points,triangles,values),
    'Classical P2 concentration':(points,triangles,reference_values),
    'Signed MHM error':(points,triangles,values-truth),
    'Signed classical error':(points,triangles,reference_values-truth),
    'MHM minus classical':(points,triangles,values-reference_values),
},figsize=(12,8))
output=Path('build/tutorial-methods');output.mkdir(parents=True,exist_ok=True)
figure.savefig(output/'transient-spatial-fields.png',dpi=200,metadata={'Author':'IPES Research Group','License':'CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/'})
plt.show()
print('Condensed global compatibility:',residual)
print('Independent classical reference:',classical)
assert residual<=1e-10


## 4. Refine space and check the independent time control

For the smooth compatible P1-trace/P2-local realization, Araya et al. (2024), assumption (A2), Theorems 2 and 3, equation (23), estimates broken energy order two. The required local refinement is present. The concentration L2 order three is additional measured accuracy, rather than a separately proved rate in that theorem. The first-order temporal integrator is a separate statement. Use the final four spatial levels and three successive rates, and inspect normalized errors $E/H^q$ as well as slopes.

The full spatial sequence is $n=4,8,16,24,32,48$. Its final three gradient orders are 1.9969, 1.9979 and 1.9986; observed concentration orders are 2.9777, 2.9846 and 2.9893. On n48, halving the time step changes the concentration and gradient errors by relative amounts below $2.5\times10^{-11}$ and $1.3\times10^{-12}$, respectively.

The `raw_residual` printed above is condensed global compatibility. The separately attributed original-row controls check every uncondensed local concentration block and global weak row at every step on n4, n16 and n48. Their algebraic checks supplement the physical errors; they are distinct diagnostics.

The independently assembled conforming P2 Galerkin reference uses the displayed `volume_forms`, source and boundary conditions on global resolutions 32, 64 and 128. It has no local/global MHM coupling. Its analytical errors decrease with concentration order three and gradient order two; the refined reference remains a numerical approximation.


In [ ]:
import json
from examples.tutorial_convergence import refinement_series,plot_method_series,asymptotic_summary
record_path=ROOT/'examples/results/tutorial-methods/transient-spatial-asymptotic.json'
record=json.loads(record_path.read_text())
series=refinement_series(
    record_path,record['rows'],'H',
    {'concentration L2':'concentration_l2','broken gradient L2':'broken_gradient_l2'},
    {'broken gradient L2':2.0},method='transient-transport',
    spaces='Local P2/r2, face P1, capacity and diffusion one, fixed smooth convection/reaction',
    rate_provenance='Araya et al. (2024), A2 and Theorems2/3: energy order two; concentration order three observed',
    root=ROOT,
)
print(asymptotic_summary(series))
figure=plot_method_series(series)
plt.show()


## References

- Harder, Paredes and Valentin (2015), *On a Multiscale Hybrid-Mixed Method for Advective-Reactive Dominated Problems with Heterogeneous Coefficients*, [DOI:10.1137/130938499](https://doi.org/10.1137/130938499).
- Araya, Jaillet, Paredes and Valentin (2024), *Generalizing the multiscale hybrid-mixed method for reactive-advective-diffusive equations*, [DOI:10.1016/j.cma.2024.117089](https://doi.org/10.1016/j.cma.2024.117089).

This manufactured square problem is a smooth verification of the stated realization, not a reproduction of a heterogeneous literature application.
